<img src="datos/img/logo_curso.png" width="450">

<p style="font-family: Arial; color: navy; text-align: center; font-size: 13px; letter-spacing: 1px; text-transform: uppercase; margin-bottom: 0;">
WSP — Python aplicado a modelos actuariales
</p>

<h1 style="background-color:#0070C0; color:white; text-align:center; font-family:Arial; padding:18px 0; border-radius:6px; margin-top:6px;">
Sesión 6 — Riesgo de reserva, cópulas y solvencia
</h1>

<div style="outline: 2px solid #EFB400; color:#000000; font-family: Arial; padding: 14px 18px; border-radius: 6px; margin-top: 14px;">
<h3 style="margin-top:0;">🎯 Objetivos</h3>
<ul>
<li>Construir pipelines de limpieza reutilizables con <code>sklearn.pipeline</code>, sobre datos como los que trabajaste en la <b>Sesión 2</b> (Calidad de datos).</li>
<li>Simular la distribución del riesgo de reserva con bootstrap ODP (<code>chainladder.BootstrapODPSample</code>) sobre triángulos como los de la <b>Sesión 3</b> (Triángulos e IBNR).</li>
<li>Retomar el riesgo de prima simulado en la <b>Sesión 5</b> y agregarlo con el riesgo de reserva bajo distintos supuestos de dependencia (suma directa, cópula independiente, Clayton, t-Student).</li>
<li>Calcular el requerimiento de capital de solvencia (SCR) y el ratio de solvencia de la compañía.</li>
</ul>
</div>

<div style="outline: 2px solid #0070C0; color:#000000; font-family: Arial; padding: 14px 18px; border-radius: 6px; margin-top: 12px;">
<h3 style="margin-top:0;">✍️ Cómo usar este notebook</h3>
<p>Los huecos que debes completar están marcados con <code>***</code>. Reemplázalos por el código o valor correspondiente y ejecuta la celda. El notebook <b>solucionario</b> (<code>solucionarios/sesion_06_riesgo_solvencia_solucionario.ipynb</code>) tiene la respuesta completa de cada <code>***</code>, con la misma estructura de celdas que este notebook. Ejecuta las celdas en orden: varias reutilizan variables definidas en celdas anteriores.</p>
</div>


## Requisitos

Este notebook usa un entorno **ya preparado** (no ejecutar `!pip install` aqui).

Paquetes necesarios (ver `requirements.txt` del curso):

- `pandas`, `numpy`, `scipy`
- `matplotlib`
- `chainladder` (0.8.24)

Si trabajas en Google Colab y el entorno no los tiene instalados, corre en una celda aparte:

```python
%pip install -r requirements.txt
```

Los datos se leen de forma local desde la carpeta `datos/` del curso (no se descargan de Drive).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import chainladder as cl
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Datos locales del curso (sin gdown / Google Drive)
DATOS = Path("../datos") if Path("../datos").exists() else Path("datos")

# Opciones de visualización
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:,.2f}".format)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("chainladder:", cl.__version__)


#🧩 ¿Qué es una Pipeline?

Una pipeline (tubería o flujo de trabajo) es una **secuencia ordenada** de pasos de procesamiento de datos donde la salida de un paso se convierte en la entrada del siguiente.

Su objetivo es automatizar y estandarizar procesos repetitivos (limpieza, transformación, modelado, predicción, etc.) para que sean reproducibles, trazables y fáciles de mantener.

Una pipeline permite:

- standarizar el flujo de trabajo (de datos crudos → resultados).

- Evitar errores humanos al aplicar transformaciones manuales.

- Reutilizar el mismo flujo sobre distintos datasets.

- Documentar de forma clara cómo se procesaron los datos.


```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# Definición del flujo
pipe = Pipeline([
    ('scaler', StandardScaler()),       # Paso 1: normalización
    ('model', LinearRegression())       # Paso 2: modelo
])

# Ajuste y predicción
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

```

> “Una pipeline es una receta: defines los pasos una sola vez, y luego puedes aplicarla a cualquier proceso similar.”

> 🔗 **Conexión:** Esta limpieza de fechas y montos es la misma que trabajaste en la Sesión 2 (Calidad de datos); aquí la empaquetamos en un pipeline de `sklearn` reutilizable.


### 🧮 Ejemplo

Supongamos que tenemos una base cruda con columnas inconsistentes y queremos:

1. Corrija las fechas.

2. Convierta los montos a numéricos.

3. Estandarice la moneda a soles.

4. Elimine filas vacías.

In [ ]:
raw_data = pd.DataFrame({
    'Fecha_Siniestro': ['2024/01/05', '2024-01-05', '2024.01.05', None],
    'Monto': ['1,200', '2,500', '-', '3,000'],
    'Moneda': ['S/', 'USD', 'S/', 'S/']
})
raw_data


In [ ]:
from dateutil import parser

def normalizar_fechas(df, columna, fill_strategy='mode'):
    fechas_normalizadas = []

    # Intentamos parsear cada valor
    for val in df[columna]:
        try:
            fechas_normalizadas.append(***)
        except (ValueError, TypeError):
            fechas_normalizadas.append(None)

    df[columna] = fechas_normalizadas

    # --- Imputación según estrategia ---
    if fill_strategy == 'mode':  # usa la más frecuente
        moda = df[columna].***()[0]
        df[columna] = df[columna].fillna(moda)
    elif fill_strategy == 'first':  # usa la primera no nula
        primera = df[columna].dropna().iloc[0]
        df[columna] = df[columna].fillna(primera)
    elif fill_strategy == 'today':  # usa la fecha actual
        df[columna] = df[columna].fillna(pd.Timestamp.today().date())

    return df


In [ ]:
def limpiar_montos(df):
    df['Monto'] = (df['Monto']
                   .replace('-', 0)
                   .replace(***, '', regex=True)
                   .astype(float))
    return df

def convertir_moneda(df, tipo_cambio=3.8):
    df.loc[df['Moneda'] == 'USD', 'Monto'] *= tipo_cambio
    df['Moneda'] = 'S/'
    return df

def eliminar_nulos(df):
    return df.dropna(subset=[***])

# --- Definimos la pipeline ---
def pipeline_limpieza(df):
    pasos = [
         lambda df: normalizar_fechas(df, 'Fecha_Siniestro', fill_strategy='mode'),
         limpiar_montos, convertir_moneda, ***]
    for paso in pasos:
        df = paso(df)
    return df

# --- Aplicamos la pipeline ---
df_limpio = pipeline_limpieza(raw_data)
df_limpio


# 💰 Riesgo de Reserva

## 🧩 Definición

El **riesgo de reserva** (o *Reserve Risk*) es la **incertidumbre asociada al valor futuro de las obligaciones** derivadas de siniestros ocurridos (ya reportados o no reportados), cuyo monto final y fecha de pago son aún desconocidos.

En otras palabras:

> Es el riesgo de que las reservas técnicas registradas hoy sean **insuficientes** frente al verdadero costo final de los siniestros.

---

## 🧱 Conexión con Solvencia II y gestión de riesgos

Bajo **Solvencia II**, el riesgo de reserva forma parte del **riesgo de  No Vida**.  
El capital requerido para cubrirlo (SCR – *Solvency Capital Requirement*) se estima considerando la distribución de los resultados de reserva con un **nivel de confianza del 99.5%** a un año.

> En términos prácticos, el riesgo de reserva refleja la **incertidumbre residual** sobre las obligaciones pasadas,  
> mientras que el **riesgo de prima** refleja la incertidumbre sobre los siniestros futuros.

> 🔗 **Conexión:** El triángulo y los factores de desarrollo son los mismos conceptos de la Sesión 3 (Triángulos e IBNR); aquí los usamos para simular la distribución de la reserva, no solo su valor central.


## 🧪 Anexo opcional: bootstrap manual paso a paso

La celda anterior usó `cl.BootstrapODPSample` dentro de un `Pipeline` de `chainladder` para simular
triángulos y obtener la distribución del riesgo de reserva. Esa clase automatiza un procedimiento de
**bootstrap sobre residuos de Pearson (ODP bootstrap)** que, internamente, hace lo siguiente:

1. Construye los triángulos incremental y acumulado.
2. Calcula los factores de desarrollo (LDF) y el triángulo "teórico" ajustado.
3. Calcula los residuos de Pearson y los reescala por grados de libertad.
4. Remuestrea esos residuos miles de veces para generar nuevos triángulos incrementales simulados.
5. Vuelve a estimar los factores de desarrollo y los *ultimates* en cada simulación.
6. Añade el **error de proceso** (ruido Gamma) para obtener la distribución final de reservas.

Si quieres ver ese procedimiento implementado a mano, con NumPy y pandas (sin usar `BootstrapODPSample`),
revisa el notebook opcional **`anexos/anexo_bootstrap_manual.ipynb`**. Es material de lectura complementaria:
no es necesario resolverlo para seguir el resto del curso.



> RR con `ChainLadder`



In [ ]:
raa = cl.load_sample('raa')
raa




> En la librería `Chainladder` tambien podemos armar procesos utilizando `pipelines`

> **Puente con el ejemplo anterior:** así como el `Pipeline` de `sklearn` encadenaba `scaler -> model` y se ajustaba con un único `.fit()`, el `Pipeline` de `chainladder` encadena `sample -> dev -> tail -> model` (remuestreo bootstrap -> factores de desarrollo -> cola -> chain ladder). La filosofía es la misma: cada paso transforma la salida del anterior, y todo el flujo se ajusta con una sola llamada a `pipe.fit(...)`.

In [ ]:
pipe = cl.Pipeline(
    steps=[
    ('sample', cl.BootstrapODPSample(random_state=42,n_sims=***)),
    ('dev', cl.Development(average=***)),
    ('tail',  cl.TailConstant(***)),
    ('model', cl.Chainladder())])

pipe.***(raa)




> Es posible acceder a los distintos ***steps*** del flujo:



In [ ]:
triangulos_simulados = pipe.named_steps.***.resampled_triangles_
triangulos_simulados


In [ ]:
triangulos_simulados.to_frame()


In [ ]:
triangulos_simulados.iloc[0]





> Y a los métodos:



In [ ]:
ultimate_simulados = pipe.named_steps.***.ultimate_
ultimate_simulados


In [ ]:
ultimate_simulados.to_frame()


In [ ]:
ultimate_simulados.iloc[0]




> Tenemos que obtener un vector de ultimates totales:



In [ ]:
rr_ultimate = ultimate_simulados.to_frame().sum(axis=***).reset_index(drop=True)
rr_ultimate




> Métricas de riesgo



In [ ]:
# --- Métricas + VaR/TVaR + riesgo de reserva ---

ES = rr_ultimate.mean()
SD = rr_ultimate.std()
alpha = ***
VaR = np.percentile(rr_ultimate, alpha*100)
TVaR = rr_ultimate[rr_ultimate > VaR].***()
riesgo_reserva = *** - ES

print("== Monte Carlo – Pérdida agregada por año ==")
print(f"E[S]     : {ES:,.2f}")
print(f"SD[S]    : {SD:,.2f}")
print(f"VaR  {int(alpha*100)}% : {VaR:,.2f}")
print(f"TVaR {int(alpha*100)}% : {TVaR:,.2f}")
print(f"Riesgo de reserva (TVaR - E[S]): {riesgo_reserva:,.2f}")




> Ya tenemos el riesgo de reserva. Necesitamos el riesgo de prima:

> 🔗 **Conexión:** El cálculo de VaR/TVaR sobre `rp_vector` es el mismo procedimiento Monte Carlo que usaste en la Sesión 5 para el riesgo de prima.


In [ ]:
rp_vector = pd.read_csv(DATOS / ***).head(1000).squeeze()
rp_vector


In [ ]:
ES = rp_vector.mean()
SD = rp_vector.std()
alpha = 0.99
VaR = np.percentile(rp_vector, ***)
TVaR = rp_vector[rp_vector > VaR].mean()
riesgo_prima = TVaR - ***

print("== Monte Carlo – Pérdida agregada por año ==")
print(f"E[S]     : {ES:,.2f}")
print(f"SD[S]    : {SD:,.2f}")
print(f"VaR  {int(alpha*100)}% : {VaR:,.2f}")
print(f"TVaR {int(alpha*100)}% : {TVaR:,.2f}")
print(f"Riesgo de prima (TVaR - E[S]): {riesgo_prima:,.2f}")


# ⚙️ Agregación de Riesgos

## 🧩 Concepto general

La **agregación de riesgos** es el proceso mediante el cual se combinan dos o más fuentes de riesgo, por ejemplo, distintas líneas de negocio, ramos, o componentes como el riesgo de prima y el riesgo de reserva para obtener la **distribución total de pérdidas** del portafolio.

Formalmente, si $ X_1, X_2, ..., X_n $ representan las pérdidas aleatorias de cada riesgo individual:

$
S = X_1 + X_2 + \dots + X_n
$

Nuestro objetivo es estimar la distribución de $S$, su valor esperado $E[S]$, y medidas de riesgo asociadas como $\text{VaR}_\alpha(S)$ y $\text{TVaR}_\alpha(S)$.

---

## 🎯 Importancia

La agregación permite cuantificar:

- El **beneficio de diversificación** entre líneas o componentes.
- La **exposición total al riesgo extremo** (colas conjuntas).
- El **capital económico** necesario (por ejemplo, bajo Solvencia II con un nivel de confianza de 99.5%).
- La **interdependencia** entre reservas, siniestros y otras variables.

En términos prácticos:

> La agregación convierte la incertidumbre de varios riesgos individuales  
> en una **visión integrada del portafolio total**.

---

Independiente

Copula Clayton

Copula Tstudent

Matriz de correlacion


## ➕ Suma directa de los vectores de riesgo

Antes de introducir las estructuras de dependencia, podemos observar qué sucede si **simplemente sumamos** los dos riesgos:
el riesgo de **prima** (`rp_vector`) y el riesgo de **reserva** (`rr_ultimate`).

Que pasa si sumamos ambos vectores?

In [ ]:
riesgo_agregado_sum = rp_vector + rr_ultimate
riesgo_agregado_sum


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(rp_vector, rr_ultimate, alpha=0.2)
plt.title("Dispersión conjunta: riesgo de prima vs. riesgo de reserva (datos originales)")
plt.xlabel("Riesgo de prima")
plt.ylabel("Riesgo de reserva")
plt.grid(True)
plt.show()


In [ ]:
ES = riesgo_agregado_sum.mean()
SD = riesgo_agregado_sum.std()
alpha = 0.99
VaR = np.percentile(riesgo_agregado_sum, alpha*100)
TVaR = riesgo_agregado_sum[riesgo_agregado_sum > VaR].mean()
riesgo_agg = *** - ***

print("== Monte Carlo – Pérdida agregada por año ==")
print(f"E[S]     : {ES:,.2f}")
print(f"SD[S]    : {SD:,.2f}")
print(f"VaR  {int(alpha*100)}% : {VaR:,.2f}")
print(f"TVaR {int(alpha*100)}% : {TVaR:,.2f}")
print(f"Riesgo agregado (TVaR - E[S]): {riesgo_agg:,.2f}")


## 🔗 Cópula Independiente

---

La **cópula independiente** asume que los riesgos no tienen relación alguna entre sí:
los valores altos o bajos de un riesgo **no influyen** en los del otro.

$
C(u,v) = uv
$

Esto equivale a decir que los riesgos de **prima** y **reserva** son *estadísticamente independientes*.


In [ ]:
# Independiente

def independiente(riesgo1: pd.Series, riesgo2: pd.Series) -> pd.Series:
    """
    Combina dos riesgos asumiendo independencia, usando simulación
    vía percentiles aleatorios (tipo copula independiente).

    riesgo1, riesgo2: Series de pérdidas simuladas o históricas
    return: Series con el riesgo agregado independiente
    """
    np.random.seed(50)
    # Nos aseguramos de que sean arrays 1D
    x = riesgo1.to_numpy().ravel()
    y = riesgo2.to_numpy().ravel()

    n = len(x)
    if len(y) != n:
        raise ValueError("riesgo1 y riesgo2 deben tener la misma longitud")

    # U(0,1) independientes
    u = np.random.rand(n)
    v = np.random.rand(n)

    # Usamos 1-u y 1-v (colas superiores)
    u_comp = 1 - ***
    v_comp = 1 - ***

    # Percentiles aleatorios de cada riesgo
    R1 = np.percentile(x, u_comp * 100)
    R2 = np.percentile(y, ***)

    riesgo_combinado = R1 + R2

    # Devolvemos una Serie
    return pd.Series(riesgo_combinado, name="riesgo_independiente")


In [ ]:
riesgo_agregado_ind = independiente(rp_vector,rr_ultimate)
riesgo_agregado_ind


In [ ]:
ES = riesgo_agregado_ind.mean()
SD = riesgo_agregado_ind.std()
alpha = 0.99
VaR = np.percentile(riesgo_agregado_ind, alpha*100)
TVaR = riesgo_agregado_ind[riesgo_agregado_ind > VaR].mean()
riesgo_agg = TVaR - ES

print("== Monte Carlo – Pérdida agregada por año ==")
print(f"E[S]     : {ES:,.2f}")
print(f"SD[S]    : {SD:,.2f}")
print(f"VaR  {int(alpha*100)}% : {VaR:,.2f}")
print(f"TVaR {int(alpha*100)}% : {TVaR:,.2f}")
print(f"Riesgo agregado (TVaR - E[S]): {riesgo_agg:,.2f}")


## 🔗 Cópula de Clayton

---

La **cópula de Clayton** introduce **dependencia en la cola inferior**, es decir,
aumenta la probabilidad de que ocurran **pérdidas altas simultáneamente** en ambos riesgos.Donde el parámetro $\theta$ (relacionado con el parámetro $ \alpha $ en nuestra implementación) controla la **intensidad de la dependencia**:

- $\theta \to  0 $ ⇒ independencia  
- $\theta \to \infty $ ⇒ dependencia perfecta  

---


In [ ]:
# Clayton

def clayton(riesgo1: pd.Series, riesgo2: pd.Series, alpha: float) -> pd.Series:
    """
    Combina dos riesgos usando una cópula de Clayton.

    riesgo1, riesgo2 : pd.Series con simulaciones o datos históricos
    alpha            : parámetro en (0,1) para la dependencia de cola
                       (theta = -log(2)/log(alpha))

    Devuelve:
        pd.Series con el riesgo agregado (Clayton)
    """
    # Transformamos alpha al parámetro theta de la cópula
    theta = -np.log(2) / np.log(***)

    # Pasamos Series -> arrays 1D
    x = riesgo1.to_numpy().ravel()
    y = riesgo2.to_numpy().ravel()

    n = len(x)
    if len(y) != n:
        raise ValueError("riesgo1 y riesgo2 deben tener la misma longitud")

    # U(0,1) independientes
    u = np.random.rand(n)
    q = np.random.rand(n)

    # Transformación de Clayton (inversa condicional)
    v_transformed = (1 + ((q ** (-(theta / (theta + 1)))) - 1) * (u ** (***))) ** (-1 / theta)

    # Complementos (colas)
    u_complement = 1 - ***
    v_complement = 1 - v_transformed

    # Percentiles aleatorios de cada riesgo
    R1 = np.percentile(x, u_complement * 100)
    R2 = np.percentile(y, v_complement * 100)

    riesgo_combinado = R1 + R2

    return pd.Series(riesgo_combinado, name="riesgo_clayton")


In [ ]:
alpha = 0.8
riesgo_agregado_clayton = clayton(rp_vector,rr_ultimate,alpha)
riesgo_agregado_clayton


In [ ]:
ES = riesgo_agregado_clayton.mean()
SD = riesgo_agregado_clayton.std()
alpha = 0.99
VaR = np.percentile(riesgo_agregado_clayton, alpha*100)
TVaR = riesgo_agregado_clayton[riesgo_agregado_clayton > VaR].mean()
riesgo_agg = TVaR - ES

print("== Monte Carlo – Pérdida agregada por año ==")
print(f"E[S]     : {ES:,.2f}")
print(f"SD[S]    : {SD:,.2f}")
print(f"VaR  {int(alpha*100)}% : {VaR:,.2f}")
print(f"TVaR {int(alpha*100)}% : {TVaR:,.2f}")
print(f"Riesgo agregado (TVaR - E[S]): {riesgo_agg:,.2f}")


In [ ]:
def simular_clayton(alpha, n=3000):
    """Genera pares (u,v) según una cópula de Clayton usando el parámetro alpha."""
    theta = -np.log(2) / np.log(alpha)
    u = np.random.rand(n)
    q = np.random.rand(n)
    v = (1 + ((q ** (-(theta / (theta + 1)))) - 1) * (u ** (-theta))) ** (-1 / theta)
    return u, v, theta

# Valores de alpha: a menor alpha -> mayor dependencia de cola
alphas = [0.95, 0.8, 0.6, 0.4]
fig, axes = plt.subplots(1, len(alphas), figsize=(15, 4))

for i, a in enumerate(alphas):
    u, v, theta = simular_clayton(a)
    u_c, v_c = 1 - u, 1 - v
    axes[i].scatter(u_c, v_c, s=10, alpha=0.4, color="steelblue")
    axes[i].set_title(f"α = {a} → θ = {theta:.2f}")
    axes[i].set_xlabel("u_complement")
    axes[i].set_ylabel("v_complement")
    axes[i].set_xlim(0, 1)
    axes[i].set_ylim(0, 1)
    axes[i].grid(True)

plt.suptitle("Cópula de Clayton – Efecto del parámetro α en la dependencia de cola", fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


## 🔗 Cópula t-Student

---

La **cópula t-Student** permite modelar **dependencia simétrica en las colas**:

no solo los eventos extremos simultáneos de grandes pérdidas (cola superior),  
sino también los de ganancias o resultados favorables (cola inferior).

In [ ]:
# T - Student

from scipy.stats import t as student_t

def tstudent(riesgo1: pd.Series, riesgo2: pd.Series, rho: float, nu: float = 4.0) -> pd.Series:
    """
    Combina dos riesgos usando una cópula t-Student con correlación rho.

    riesgo1, riesgo2 : pd.Series
        Series de datos simulados o históricos de cada riesgo.
    rho : float
        Correlación entre los riesgos (en [-1, 1]).
    nu : float
        Grados de libertad de la distribución t (menor = colas más pesadas).

    Devuelve:
        pd.Series con el riesgo agregado dependiente.
    """

    # Convertir a arrays
    x = riesgo1.to_numpy().ravel()
    y = riesgo2.to_numpy().ravel()

    n = len(x)
    if len(y) != n:
        raise ValueError("riesgo1 y riesgo2 deben tener la misma longitud")

    # Matriz de covarianza
    cov = [[1, ***],
           [rho, 1]]

    # Generación de muestras t-Student bivariadas
    z = np.random.multivariate_normal(mean=[0, 0], cov=cov, size=n)
    chi2 = np.random.chisquare(df=nu, size=n)
    w = np.sqrt(chi2 / ***)
    t_samples = z / w[:, None]  # escala por raíz del chi-cuadrado

    T1, T2 = t_samples[:, 0], t_samples[:, 1]

    # Transformar a escala uniforme (CDF de la t)
    u = student_t.cdf(T1, df=nu)
    v = student_t.***(T2, df=nu)

    # Combinar riesgos según los percentiles simulados
    R1 = np.percentile(x, u * 100)
    R2 = np.percentile(y, v * 100)
    riesgo_combinado = R1 + R2

    return pd.Series(riesgo_combinado, name="riesgo_tstudent")


In [ ]:
riesgo_agregado_tstudent = tstudent(rp_vector,rr_ultimate,rho = 0.5)
riesgo_agregado_tstudent


In [ ]:
ES = riesgo_agregado_tstudent.mean()
SD = riesgo_agregado_tstudent.std()
alpha = 0.99
VaR = np.percentile(riesgo_agregado_tstudent, alpha*100)
TVaR = riesgo_agregado_tstudent[riesgo_agregado_tstudent > VaR].mean()
riesgo_agg = TVaR - ES

print("== Monte Carlo – Pérdida agregada por año ==")
print(f"E[S]     : {ES:,.2f}")
print(f"SD[S]    : {SD:,.2f}")
print(f"VaR  {int(alpha*100)}% : {VaR:,.2f}")
print(f"TVaR {int(alpha*100)}% : {TVaR:,.2f}")
print(f"Riesgo agregado (TVaR - E[S]): {riesgo_agg:,.2f}")


In [ ]:
def simular_uv_tstudent(rho, nu=4.0, n=3000):
    cov = [[1, rho], [rho, 1]]
    z = np.random.multivariate_normal(mean=[0, 0], cov=cov, size=n)
    chi2 = np.random.chisquare(df=nu, size=n)
    w = np.sqrt(chi2 / nu)
    t_samples = z / w[:, None]
    T1, T2 = t_samples[:, 0], t_samples[:, 1]
    u = student_t.cdf(T1, df=nu)
    v = student_t.cdf(T2, df=nu)
    return u, v

rhos = [0.1, 0.5, 0.8, 0.95]
fig, axes = plt.subplots(1, len(rhos), figsize=(15, 4))

for i, r in enumerate(rhos):
    u, v = simular_uv_tstudent(r)
    axes[i].scatter(u, v, s=10, alpha=0.4, color="firebrick")
    axes[i].set_title(f"ρ = {r}")
    axes[i].set_xlim(0, 1)
    axes[i].set_ylim(0, 1)
    axes[i].set_xlabel("u")
    axes[i].set_ylabel("v")
    axes[i].grid(True)

plt.suptitle("Cópula t-Student – efecto de la correlación ρ", fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


> **¿Por qué usamos el resultado de la cópula t-Student para el requerimiento de capital?**
>
> De las tres formas de combinar riesgo de prima y riesgo de reserva (independiente, Clayton, t-Student),
> la cópula **t-Student** es la que mejor captura la **dependencia simétrica de colas**: refleja que,
> en escenarios extremos, ambos riesgos tienden a materializarse **juntos** (no solo en la cola inferior,
> como en Clayton). Por eso se usa como insumo (`riesgo_no_vida`) en el cálculo de solvencia.

In [ ]:
# --- Riesgo agregado final (cópula t-Student) usado para el SCR ---

ES = riesgo_agregado_tstudent.mean()
SD = riesgo_agregado_tstudent.std()
alpha = 0.99
VaR = np.percentile(riesgo_agregado_tstudent, ***)
TVaR = riesgo_agregado_tstudent[riesgo_agregado_tstudent > VaR].mean()
riesgo_agg = *** - ES

print("== Monte Carlo – Pérdida agregada por año (t-Student) ==")
print(f"E[S]     : {ES:,.2f}")
print(f"SD[S]    : {SD:,.2f}")
print(f"VaR  {int(alpha*100)}% : {VaR:,.2f}")
print(f"TVaR {int(alpha*100)}% : {TVaR:,.2f}")
print(f"Riesgo agregado t-Student (TVaR - E[S]): {riesgo_agg:,.2f}")


 🧾 Solvencia y Ratio de Solvencia

---

## 💡 Concepto general

La **solvencia** de una entidad aseguradora refleja su **capacidad para hacer frente a sus obligaciones** con los asegurados y terceros en todo momento.  
Es un pilar fundamental del marco regulatorio **Solvencia II**, que busca garantizar la estabilidad del sistema financiero y la protección del asegurado.

---

## ⚙️ Ratio de Solvencia

El **ratio de solvencia** mide el grado de suficiencia del capital disponible frente al capital exigido por el regulador.

$
\text{Ratio de Solvencia} = \frac{\text{Fondos Propios Admisibles}}{\text{Requerimiento de Capital}} = \frac{\text{FP}}{\text{SCR}}
$

Donde:
- **FP** = Fondos propios admisibles (*Own Funds*)  
- **SCR** = Requerimiento de capital de solvencia (*Solvency Capital Requirement*)

---

### 📏 Interpretación

| Ratio | Interpretación |
|--------|----------------|
| **> 1.00 (100%)** | Capital suficiente para cubrir el requerimiento regulatorio. |
| **≈ 1.00** | Nivel justo de solvencia: sin margen adicional. |
| **< 1.00** | Insuficiencia de capital: incumplimiento regulatorio y posible intervención. |

En general, las aseguradoras suelen mantener un **margen adicional de seguridad**, por ejemplo un ratio entre **130% y 180%**, para absorber la volatilidad de los riesgos de mercado, suscripción y operación.

---

In [ ]:
data = {
    'Concepto': [
        'Caja Banco', 'Bonos', 'Inmuebles', 'Reservas de riesgos en curso',
        'Reservas de siniestros', 'Reservas matemáticas', 'Cuentas por pagar'
    ],
    'Monto ($)': [
        1000000, 45000000, 10000000, 1000000,
        '-', '-', 500000
    ]
}

# Creando el DataFrame
balance_general = pd.DataFrame(data)
balance_general


In [ ]:
riesgo_agregado_vida = 890_375


In [ ]:
# Matriz de correlacion
correlation_matrix = pd.DataFrame(np.array([
    [1.00, 0.25, 0.50, 0.25],
    [0.25, 1.00, 0.25, 0.25],
    [0.25, 0.25, 1.00, 0.25],
    [0.25, 0.25, 0.25, 1.00]
]))

riesgo_vida = ***

riesgo_no_vida = ***

riesgo_mercado = 1500000

riesgo_contraparte = 300000

riesgo_operacional = 1200000

fondos_propios =  4000000


In [ ]:
matriz_riesgos = pd.DataFrame(np.array([
    [riesgo_vida],
    [riesgo_no_vida],
    [riesgo_mercado],
    [riesgo_contraparte],
]))

matriz_riesgos


In [ ]:
riesgo_agregado = np.sqrt(matriz_riesgos.T.dot(correlation_matrix.dot(***)))
riesgo_agregado


In [ ]:
requerimiento = riesgo_agregado + ***
requerimiento


In [ ]:
rsol = fondos_propios/***
rsol


## ✅ Resumen

En este notebook:

1. Calculamos el **riesgo de reserva** simulando triángulos con `cl.BootstrapODPSample` dentro de un `Pipeline` de `chainladder`.
2. Calculamos el **riesgo de prima** a partir de un vector de pérdidas simuladas.
3. Agregamos ambos riesgos de cuatro formas (suma directa, cópula independiente, Clayton, t-Student) y comparamos el efecto de cada supuesto de dependencia sobre el `TVaR`.
4. Usamos el resultado de la cópula t-Student para construir el **requerimiento de capital de solvencia (SCR)** y el **ratio de solvencia**.

> Para ver **qué hace `BootstrapODPSample` por dentro** (triángulos incremental/acumulado, residuos de Pearson, remuestreo, error de proceso), revisa el anexo opcional `anexos/anexo_bootstrap_manual.ipynb`.